  **O que este script faz?**



  1. **Leitura de Dados (Extract):** Conecta ao Google BigQuery e carrega tabelas da camada Bronze contendo dados de bilheteria e avaliações de filmes.



  2. **Integração de Dados (Transform):** Realiza cruzamento (Merge/Join) entre datasets distintos para consolidar informações financeiras, técnicas e de avaliação dos filmes da Marvel e DC.



  3. **Tratamento e Padronização:** Limpa colunas monetárias, percentuais, datas e métricas numéricas, garantindo consistência analítica para futuras consultas e dashboards.



  4. **Normalização de Strings:** Simplifica campos textuais complexos (gênero, idioma, produtoras etc.) mantendo apenas o primeiro valor principal de cada lista textual.



  5. **Carga Analítica (Load):** Salva o dataset tratado e consolidado em uma camada "Silver" no Google BigQuery, representando dados limpos e prontos para análise.

In [ ]:
# ==============================================================================
# 1. IMPORTAÇÃO DE BIBLIOTECAS E DEPENDÊNCIAS
# ==============================================================================

# Ferramentas de Nuvem (Google Cloud Platform - GCP)
from google.cloud import bigquery  # Cliente oficial de comunicação com o Google BigQuery.
from google.oauth2 import service_account  # Responsável pela autenticação segura via Service Account JSON.

# Manipulação e Análise de Dados
import pandas as pd  # Biblioteca principal para tratamento tabular e análises de dados.

In [ ]:
# ==============================================================================
# 2. CONFIGURAÇÕES DE VISUALIZAÇÃO (PANDAS DISPLAY CONFIG)
# ==============================================================================

# Por padrão, o Pandas esconde colunas/linhas se a tabela for muito grande.
# Estas configurações garantem que possamos inspecionar os dados no terminal/notebook.
pd.set_option("display.max_columns", None)  # Não limita o número máximo de colunas exibidas na tela.
pd.set_option("display.max_rows", 10)  # Limita a exibição visual a no máximo 10 linhas para não travar a IDE.

In [ ]:
# ==============================================================================
# 3. CONFIGURAÇÃO DE AUTENTICAÇÃO E CONEXÃO COM O BIGQUERY
# ==============================================================================

# Caminho local do arquivo JSON contendo as credenciais privadas da conta de serviço.
# ATENÇÃO: Nunca versionar este arquivo no GitHub público.
meu_service_account_file = "../secrets/_______________________.json"

# ID do projeto pessoal criado no Google Cloud Platform.
meu_gcp_project_id = "_______________________"

# Conversão do arquivo JSON em um objeto de credenciais autenticadas.
gcp_credentials = service_account.Credentials.from_service_account_file(meu_service_account_file)

# Inicialização do cliente de comunicação com o BigQuery.
# Este objeto será utilizado para leitura e escrita de tabelas na nuvem.
bigquery_client = bigquery.Client(credentials=gcp_credentials, project=meu_gcp_project_id)

In [ ]:
# ==============================================================================
# 4. EXTRAÇÃO DE DADOS (EXTRACT)
# ==============================================================================

# ------------------------------------------------------------------------------
# Fonte: Camada Bronze - BigQuery
# Objetivo:
#   Carregar dados financeiros e comerciais de filmes da Marvel e DC.
#
# Dataset:
#   bronze.boxoffice_dc_marvel
# ------------------------------------------------------------------------------

df_boxoffice_dc_marvel = bigquery_client.query("""
    SELECT *
    FROM bronze.boxoffice_dc_marvel
    """).to_dataframe(create_bqstorage_client=False)

In [ ]:
# ------------------------------------------------------------------------------
# Fonte: Camada Bronze - BigQuery
# Objetivo:
#   Carregar dados técnicos e avaliações do IMDB.
#
# Dataset:
#   bronze.imdb_filmes
# ------------------------------------------------------------------------------

df_imdb_filmes = bigquery_client.query("SELECT * FROM bronze.imdb_filmes ").to_dataframe(create_bqstorage_client=False)

In [ ]:
# ==============================================================================
# 5. CONSOLIDAÇÃO DOS DATASETS (MERGE / JOIN)
# ==============================================================================

# Realizando junção entre os datasets financeiros e técnicos.
#
# Regras de cruzamento:
#   - FILM  -> TITLE
#   - YEAR  -> YEAR
#
# Tipo de JOIN:
#   - OUTER JOIN
#
# Objetivo:
#   Preservar todos os registros presentes em ambas as tabelas,
#   mesmo que existam inconsistências entre os datasets.

df_complete_movie_info = (
    pd.merge(
        df_boxoffice_dc_marvel,
        df_imdb_filmes,
        left_on=["FILM", "YEAR"],
        right_on=["TITLE", "YEAR"],
        how="outer",
    )
    .sort_values(by="YEAR", ascending=False)
    .drop_duplicates(keep="last")
)

In [ ]:
# ==============================================================================
# 6. FILTRAGEM DE FRANQUIAS (MARVEL E DC)
# ==============================================================================

# Mantendo apenas filmes pertencentes às franquias:
#   - Marvel
#   - DC
#
# Isso elimina registros de outros universos cinematográficos presentes no dataset.

df_complete_dc_marvel = df_complete_movie_info[
    (df_complete_movie_info["FRANCHISE"] == "Marvel") | (df_complete_movie_info["FRANCHISE"] == "DC")
]

In [ ]:
# ==============================================================================
# 7. REMOÇÃO DE COLUNAS REDUNDANTES E DESNECESSÁRIAS
# ==============================================================================

# Eliminando colunas:
#   - duplicadas após o merge,
#   - auxiliares,
#   - pouco relevantes para análises finais,
#   - ou já representadas por outras métricas equivalentes.

df_complete_dc_marvel = df_complete_dc_marvel.drop(
    columns=[
        "25X_PROD",
        "BOX_OFFICE_GROSS_DOMESTIC_US_AND_CANADA",
        "BOX_OFFICE_GROSS_WORLDWIDE",
        "BUDGET_y",
        "DURATION",
        "LENGTH",
        "MPAA_RATING",
        "TITLE",
        "YEAR",
    ]
).reset_index(drop=True)

In [ ]:
# ==============================================================================
# 8. VALIDAÇÃO EXPLORATÓRIA DOS DADOS
# ==============================================================================

# Verificando os filmes com maior nota no IMDB após o processo de consolidação.
# Esta análise rápida ajuda a validar:
#   - integridade do merge,
#   - consistência das notas,
#   - ordenação correta dos dados.

df_complete_dc_marvel.sort_values(by="RATING_IMDB", ascending=False).head(2)

In [ ]:
# ==============================================================================
# 9. CÓPIA DE SEGURANÇA PARA TRATAMENTO (DATA CLEANING)
# ==============================================================================

# Criando uma cópia independente do DataFrame consolidado.
# Isso evita modificar acidentalmente o dataset original durante os tratamentos.

df_tratado = df_complete_dc_marvel.copy()

In [ ]:
# ==============================================================================
# 10. TRATAMENTO DE COLUNAS MONETÁRIAS
# ==============================================================================

# Colunas contendo valores financeiros armazenados originalmente como texto.
#
# Problemas encontrados:
#   - símbolo "$"
#   - separadores de milhar ","
#   - valores nulos representados como string "None"
#
# Objetivo:
#   Converter tudo para tipo numérico float.

money_cols = [
    "BOX_OFFICE_GROSS_OTHER_TERRITORIES",
    "BUDGET_x",
    "INFLATION_ADJUSTED_BUDGET",
    "INFLATION_ADJUSTED_WORLDWIDE_GROSS",
    "GROSS_OPENING_WEEKEND",
    "GROSS_US_CANADA",
    "GROSS_WORLD_WIDE",
]

for col in money_cols:
    df_tratado[col] = (
        df_tratado[col].astype(str).str.replace("$", "").str.replace(",", "").replace("None", 0).astype(float)
    )

In [ ]:
# ==============================================================================
# 11. TRATAMENTO DE COLUNAS PERCENTUAIS
# ==============================================================================

# Removendo o símbolo "%" e convertendo os valores para float.

percent_cols = ["DOMESTIC"]

for col in percent_cols:
    df_tratado[col] = df_tratado[col].astype(str).str.replace("%", "").astype(float)

In [ ]:
# ==============================================================================
# 12. PADRONIZAÇÃO DE COLUNAS DE DATA
# ==============================================================================

# Conversão de strings para formato datetime.
#
# Formato original:
#   DD/MM/YYYY

date_cols = ["US_RELEASE_DATE"]

for col in date_cols:
    df_tratado[col] = pd.to_datetime(df_tratado[col], format="%d/%m/%Y")

In [ ]:
# ==============================================================================
# 13. TRATAMENTO DE MÉTRICAS INTEIRAS
# ==============================================================================

# Colunas quantitativas contendo:
#   - votos,
#   - premiações,
#   - indicações,
#   - métricas críticas.
#
# Estratégia:
#   - preencher nulos com zero,
#   - converter para inteiro.

int_cols = [
    "VOTE",
    "WIN",
    "OSCAR",
    "NOMINATION",
    "ROTTEN_TOMATOES_CRITIC_SCORE",
]

for col in int_cols:
    df_tratado[col] = df_tratado[col].fillna(0).astype(int)

In [ ]:
# ==============================================================================
# 14. NORMALIZAÇÃO DE CAMPOS TEXTUAIS
# ==============================================================================

# Alguns campos possuem múltiplos valores separados por vírgula.
#
# Exemplo:
#   "Action, Adventure, Sci-Fi"
#
# Estratégia:
#   - manter apenas o primeiro valor principal,
#   - remover espaços excedentes.

df_tratado["GENRE"] = df_tratado["GENRE"].str.split(",").str[0].str.strip()

df_tratado["COUNTRY_ORIGIN"] = df_tratado["COUNTRY_ORIGIN"].str.split(",").str[0].str.strip()

df_tratado["LANGUAGE"] = df_tratado["LANGUAGE"].str.split(",").str[0].str.strip()

df_tratado["PRODUCTION_COMPANY"] = df_tratado["PRODUCTION_COMPANY"].str.split(",").str[0].str.strip()

df_tratado["WRITER"] = df_tratado["WRITER"].str.split(",").str[0].str.strip()

In [ ]:
# ==============================================================================
# 15. HIGIENIZAÇÃO FINAL DO DATAFRAME
# ==============================================================================

# Removendo colunas auxiliares desnecessárias.
#
# errors="ignore":
#   evita erro caso a coluna não exista no dataset.

df_tratado = df_tratado.drop(columns=["Unnamed: 0"], errors="ignore")

# Reorganizando colunas em ordem alfabética
# para padronização estrutural.

df_tratado = df_tratado[sorted(df_tratado.columns)].drop_duplicates()

In [ ]:
# ==============================================================================
# 16. CARGA ANALÍTICA - CAMADA SILVER (LOAD)
# ==============================================================================

# Salvando o dataset tratado na camada Silver do Data Warehouse.
#
# Camada Silver:
#   - dados limpos,
#   - estruturados,
#   - prontos para análises,
#   - dashboards,
#   - machine learning,
#   - ou consumo analítico.

df_tratado.to_gbq(
    project_id=meu_gcp_project_id,
    destination_table="silver.complete_dc_marvel",
    if_exists="replace",
    credentials=gcp_credentials,
)